# Лабораторна робота 4 — LASSO-регресія через координатний спуск

**Набір даних:** `kc_house_data.csv`  
**Обмеження:** scikit-learn LASSO **не дозволено** для базових завдань.

## Налаштування

In [1]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

%matplotlib inline


## Теоретичне підґрунтя

Для нормалізованих ознак значення ρᵢ для координатного спуску LASSO:
```
ρᵢ = dot( featureᵢ,  output − (Xw − wᵢ·featureᵢ) )
```
Правило м'якого порогування для wᵢ (i ≥ 1):
```
wᵢ = ρᵢ + λ/2    якщо ρᵢ < −λ/2
wᵢ = 0           якщо −λ/2 ≤ ρᵢ ≤ λ/2
wᵢ = ρᵢ − λ/2    якщо ρᵢ >  λ/2
```
Вільний член (i = 0) **не** регуляризується: w₀ = ρ₀.

## Допоміжні функції — `get_numpy_data` і `predict_output` (повторне використання з Лаб. 3)

Наведено для зручності. Якщо ви виконали Лабораторну роботу 3, можете підставити власну реалізацію.

In [3]:
def get_numpy_data(dataframe, features, output):
    data = dataframe.copy()
    data['constant'] = 1.0

    feature_columns = ['constant'] + features
    feature_matrix = data[feature_columns].to_numpy(dtype=float)
    
    output_array = data[output].to_numpy(dtype=float)

    return feature_matrix, output_array


In [4]:
def predict_output(feature_matrix, weights):
    return np.dot(feature_matrix, weights)


---
## Завдання 1 — Нормалізація ознак

Реалізуйте `normalize_features(feature_matrix)`, яка ділить кожен стовпець на його норму L2 та повертає `(normalised_features, norms)`.

In [5]:
def normalize_features(feature_matrix):
    """
    Ділить кожен стовпець на його норму L2.

    Повертає
    -------
    normalised : np.ndarray, та сама форма що й feature_matrix
    norms      : np.ndarray, shape (n_cols,)
    """
    norms = np.linalg.norm(feature_matrix, axis=0)

    safe_norms = norms.copy()
    safe_norms[safe_norms == 0] = 1.0

    normalised = feature_matrix / safe_norms
    return normalised, norms


### Перевірка — кожен стовпець `normalised` має дорівнювати [0.6, 0.8]; norms = [5, 10, 15]

In [6]:
test_matrix = np.array([[3., 6., 9.], [4., 8., 12.]])
normalised, norms = normalize_features(test_matrix)
print('Нормалізована матриця:\n', normalised)
print('Норми:', norms)


Нормалізована матриця:
 [[0.6 0.6 0.6]
 [0.8 0.8 0.8]]
Норми: [ 5. 10. 15.]


---
## Завдання 2 — Один крок координатного спуску

Реалізуйте `lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty)`. Функція має:
1. Обчислити поточне передбачення.
2. Обчислити ρᵢ.
3. Якщо i = 0: wᵢ = ρᵢ (без штрафу). Якщо i ≥ 1: застосувати правило м'якого порогування.
4. Повернути нове значення ваги.

In [7]:
def lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty):
    """
    Виконує один крок координатного спуску LASSO для ваги з індексом i.

    Параметри
    ----------
    i             : int    індекс ваги для оновлення
    feature_matrix: array  нормалізована, shape (n, d)
    output        : array  shape (n,)
    weights       : array  поточні ваги, shape (d,)
    l1_penalty    : float  λ

    Повертає
    -------
    new_weight_i  : float
    """
    prediction = predict_output(feature_matrix, weights)

    # Compute rho_i
    feature_i = feature_matrix[:, i]
    rho_i = np.dot(feature_i, output - (prediction - weights[i] * feature_i))

    # Apply soft-threshold (intercept is unregularised)
    if i == 0:   # intercept — no penalty
        new_weight_i = rho_i
    else:
        if rho_i < -l1_penalty / 2:
            new_weight_i = rho_i + l1_penalty / 2
        elif rho_i > l1_penalty / 2:
            new_weight_i = rho_i - l1_penalty / 2
        else:
            new_weight_i = 0.0

    return new_weight_i


### Перевірка — результат має бути ≈ 0.4256

In [8]:
from math import sqrt
result = lasso_coordinate_descent_step(
    i=1,
    feature_matrix=np.array([[3/sqrt(13), 1/sqrt(10)],
                             [2/sqrt(13), 3/sqrt(10)]]),
    output=np.array([1., 1.]),
    weights=np.array([1., 4.]),
    l1_penalty=0.1
)
print(f'Результат кроку: {result:.4f}  (очікується ≈ 0.4256)')


Результат кроку: 0.4256  (очікується ≈ 0.4256)


---
## Завдання 3 — Циклічний координатний спуск

Реалізуйте `lasso_cyclical_coordinate_descent(feature_matrix, output, initial_weights, l1_penalty, tolerance)`. У кожному циклі оновлюйте ваги 0, 1, …, d−1 по черзі, використовуючи оновлену вагу одразу. Зупиняйтесь, коли максимальна абсолютна зміна ваги за цикл < `tolerance`.

In [9]:
def lasso_cyclical_coordinate_descent(feature_matrix, output,
                                       initial_weights, l1_penalty, tolerance):
    """
    Запускає циклічний координатний спуск до збіжності.

    Повертає
    -------
    weights : np.ndarray — навчені ваги (на нормалізованих ознаках)
    """
    weights = np.array(initial_weights, dtype=float)

    gram_matrix = np.dot(feature_matrix.T, feature_matrix)
    feature_output = np.dot(feature_matrix.T, output)

    while True:
        max_change = 0.0

        for i in range(len(weights)):
            old_weight_i = weights[i]

            # Update weight i
            rho_i = (
                feature_output[i]
                - np.dot(gram_matrix[i, :], weights)
                + gram_matrix[i, i] * weights[i]
            )

            if i == 0:
                new_weight_i = rho_i
            else:
                if rho_i < -l1_penalty / 2:
                    new_weight_i = rho_i + l1_penalty / 2
                elif rho_i > l1_penalty / 2:
                    new_weight_i = rho_i - l1_penalty / 2
                else:
                    new_weight_i = 0.0

            weights[i] = new_weight_i
            # Track the largest change this cycle
            max_change = max(max_change, abs(new_weight_i - old_weight_i))

        if max_change < tolerance:
            break

    return weights


---
## Завдання 4 — Відбір ознак

Побудуйте нормалізовану матрицю ознак з `kc_house_data.csv`, використовуючи `['sqft_living', 'bedrooms']`. Запустіть координатний спуск з:
```
initial_weights = [0., 0., 0.]
tolerance       = 1.0
```
— `l1_penalty = 1e7`: вкажіть навчені ваги, RSS та які ознаки мають ненульові ваги.  
— `l1_penalty = 1e8`: повторіть. Що змінилося?

In [10]:
sales = pd.read_csv('kc_house_data.csv')

# Побудуйте та нормалізуйте матрицю ознак
feature_matrix, output = get_numpy_data(sales, ['sqft_living', 'bedrooms'], 'price')
normalised_features, feature_norms = normalize_features(feature_matrix)

print('Форма матриці ознак:', feature_matrix.shape)
print('Норми стовпців:', feature_norms)


Форма матриці ознак: (21613, 3)
Норми стовпців: [1.47013605e+02 3.34257264e+05 5.14075870e+02]


In [11]:
feature_names = ['intercept', 'sqft_living', 'bedrooms']

# Запуск з l1_penalty = 1e7
weights_1e7 = lasso_cyclical_coordinate_descent(
    normalised_features, output,
    initial_weights=np.zeros(3),
    l1_penalty=1e7,
    tolerance=1.0
)

preds_1e7 = predict_output(normalised_features, weights_1e7)
rss_1e7 = np.sum((preds_1e7 - output) ** 2)
non_zero_1e7 = [name for name, weight in zip(feature_names, weights_1e7) if abs(weight) > 1e-7]

print('Ваги (λ=1e7):')
for name, weight in zip(feature_names, weights_1e7):
    print(f'{name:12s}: {weight:.6f}')
print('Non-zero features:', non_zero_1e7)
print(f'RSS (λ=1e7): {rss_1e7:.3e}')


Ваги (λ=1e7):
intercept   : 21624997.959519
sqft_living : 63157247.207890
bedrooms    : 0.000000
Non-zero features: ['intercept', 'sqft_living']
RSS (λ=1e7): 1.630e+15


In [12]:
# Запуск з l1_penalty = 1e8
weights_1e8 = lasso_cyclical_coordinate_descent(
    normalised_features, output,
    initial_weights=np.zeros(3),
    l1_penalty=1e8,
    tolerance=1.0
)

preds_1e8 = predict_output(normalised_features, weights_1e8)
rss_1e8 = np.sum((preds_1e8 - output) ** 2)
non_zero_1e8 = [name for name, weight in zip(feature_names, weights_1e8) if abs(weight) > 1e-7]

print('Ваги (λ=1e8):')
for name, weight in zip(feature_names, weights_1e8):
    print(f'{name:12s}: {weight:.6f}')
print('Non-zero features:', non_zero_1e8)
print(f'RSS (λ=1e8): {rss_1e8:.3e}')


Ваги (λ=1e8):
intercept   : 79400304.637645
sqft_living : 0.000000
bedrooms    : 0.000000
Non-zero features: ['intercept']
RSS (λ=1e8): 2.913e+15


**Спостереження:**

- При `λ = 1e7` ненульовими залишилися `intercept` і `sqft_living`, а вага `bedrooms` стала нульовою. Це означає, що за такого штрафу модель залишила площу житла як корисну ознаку, але кількість спалень не дала достатньо сильного додаткового внеску.
- При `λ = 1e8` ненульовим залишився лише `intercept`; обидві ознаки (`sqft_living`, `bedrooms`) занулилися. Це сталося тому, що L1-штраф став значно сильнішим і модель стала максимально простою.
- RSS при `λ = 1e8` більший, ніж при `λ = 1e7`, бо модель втратила важливу пояснювальну ознаку `sqft_living`.


---
## ✨ Бонус — Оцінка на ненормалізованих тестових даних

Розбийте дані 80 % / 20 % (`random_state=42`). Побудуйте та нормалізуйте навчальну матрицю ознак з усіма 10 ознаками нижче. Запустіть координатний спуск (`l1_penalty=1e7`, `tolerance=1.0`).

Ознаки: `sqft_living`, `bedrooms`, `bathrooms`, `sqft_lot`, `floors`, `waterfront`, `view`, `condition`, `grade`, `yr_built`.

Щоб оцінювати на ненормалізованих тестових даних, масштабуйте ваги:
```
weights_rescaled = weights / norms
```
Обчисліть RSS на ненормалізованій тестовій матриці ознак. Вкажіть відібрані ознаки.

In [13]:
FEATURES = [
    'sqft_living', 'bedrooms', 'bathrooms', 'sqft_lot',
    'floors', 'waterfront', 'view', 'condition',
    'grade', 'yr_built'
]

train_data, test_data = train_test_split(sales, test_size=0.2, random_state=42)

# Бонус — нормалізуйте навчальні ознаки, запустіть LASSO, масштабуйте ваги, оцініть

# 1. Будуємо матрицю ознак для навчальної вибірки.
train_feature_matrix, train_output = get_numpy_data(train_data, FEATURES, 'price')

# 2. Нормалізуємо лише навчальну матрицю.
train_normalised_features, train_norms = normalize_features(train_feature_matrix)

# 3. Навчаємо LASSO через координатний спуск.
initial_weights = np.zeros(len(FEATURES) + 1)
weights_normalised = lasso_cyclical_coordinate_descent(
    train_normalised_features,
    train_output,
    initial_weights=initial_weights,
    l1_penalty=1e7,
    tolerance=1.0
)

# 4. Переводимо ваги з нормалізованого масштабу назад у звичайний масштаб.
weights_rescaled = weights_normalised / train_norms

# 5. Оцінюємо якість на тестовій вибірці без нормалізації тестових ознак.
test_feature_matrix, test_output = get_numpy_data(test_data, FEATURES, 'price')
test_predictions = predict_output(test_feature_matrix, weights_rescaled)
test_rss = np.sum((test_predictions - test_output) ** 2)

all_feature_names = ['intercept'] + FEATURES
selected_features = [
    name for name, weight in zip(all_feature_names, weights_normalised)
    if abs(weight) > 1e-7
]

print('Ваги на нормалізованих ознаках:')
for name, weight in zip(all_feature_names, weights_normalised):
    print(f'{name:12s}: {weight:.6f}')

print()
print('Ваги після повернення до початкового масштабу:')
for name, weight in zip(all_feature_names, weights_rescaled):
    print(f'{name:12s}: {weight:.6f}')

print()
print('Відібрані ознаки:', selected_features)
print(f'Тестова RSS: {test_rss:.3e}')


Ваги на нормалізованих ознаках:
intercept   : 25690961.303403
sqft_living : 46465774.654724
bedrooms    : 0.000000
bathrooms   : 0.000000
sqft_lot    : 0.000000
floors      : 0.000000
waterfront  : 2359222.478039
view        : 7693974.962696
condition   : 0.000000
grade       : 0.000000
yr_built    : 0.000000

Ваги після повернення до початкового масштабу:
intercept   : 195381.238557
sqft_living : 156.107371
bedrooms    : 0.000000
bathrooms   : 0.000000
sqft_lot    : 0.000000
floors      : 0.000000
waterfront  : 211864.432858
view        : 73456.065248
condition   : 0.000000
grade       : 0.000000
yr_built    : 0.000000

Відібрані ознаки: ['intercept', 'sqft_living', 'waterfront', 'view']
Тестова RSS: 3.425e+14


**Відібрані ознаки:** `intercept`, `sqft_living`, `waterfront`, `view`.

**Тестова RSS:** приблизно `3.425 × 10^14`.

**Інтерпретація:** вибрані ознаки мають логічний сенс. `sqft_living` відображає житлову площу, яка зазвичай сильно впливає на ціну будинку. `waterfront` показує наявність розташування біля води, що може суттєво підвищувати вартість. `view` також є важливою якісною характеристикою нерухомості. Інші ознаки за заданого L1-штрафу були занулені, тобто модель визнала їхній додатковий внесок недостатнім порівняно зі штрафом за складність.
